# 2. Stackable Knowledge Layers & Conflict Resolution

Demonstrates non-destructive `StackManager` composition (`overlay`, `union`, `intersection`, `difference`, `diff_layers`) and inspecting preserved `ConflictRecord`s.

In [ ]:
import tempfile
from pathlib import Path

from knowledgereduce import (
    Claim,
    ConflictStrategy,
    DuckDBParquetStore,
    Entity,
    Evidence,
    KnowledgeLayer,
    LayerData,
    LayerTier,
    MergeMode,
    Relation,
    ReliabilityRating,
    StackManager,
)

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    store = DuckDBParquetStore(Path(tmp) / "store")
    mgr = StackManager(store)

    base = LayerData(
        layer=KnowledgeLayer(layer_id="base", name="Base", tier=LayerTier.REDUCED),
        entities=[Entity(entity_id="ent_ch", canonical_name="Civic Honors", entity_type="ORG")],
        claims=[
            Claim(
                claim_id="clm_pub",
                statement="Civic Honors was published in 2006.",
                reliability=ReliabilityRating.LIKELY_TRUE,
                source_id="book_2006",
                evidence=[Evidence(source_id="book_2006", excerpt="published in 2006")],
            )
        ],
        relations=[Relation(relation_id="r1", source_id="clm_pub", target_id="ent_ch", relation_type="MENTIONS")],
    )
    patch = LayerData(
        layer=KnowledgeLayer(layer_id="patch", name="Web Patch", tier=LayerTier.CURATED),
        entities=[
            Entity(entity_id="ent_ch", canonical_name="Civic Honors", entity_type="ORG"),
            Entity(entity_id="ent_nels", canonical_name="Nels Lindahl", entity_type="PERSON"),
        ],
        claims=[
            Claim(
                claim_id="clm_pub",
                statement="Civic Honors was republished online in 2020.",
                reliability=ReliabilityRating.VERIFIED,
                source_id="web_2020",
                evidence=[Evidence(source_id="web_2020", excerpt="online in 2020")],
            )
        ],
        relations=[Relation(relation_id="r2", source_id="ent_nels", target_id="ent_ch", relation_type="AUTHORED")],
    )
    store.save_layer(base)
    store.save_layer(patch)

    stacked = mgr.stack_overlay("base", "patch", target_layer_id="stacked", conflict_strategy=ConflictStrategy.PRESERVE_DISPUTE)
    diff = mgr.diff_layers("base", "stacked")
    print("Conflicts:", len(stacked.conflicts), "| Added entities:", diff.added_entity_ids)
    print("Union count:", len(mgr.merge_layers(["base", "patch"], target_layer_id="u1", mode=MergeMode.UNION).entities))